# Infant State Recognition System — Phase 1
**AML Project 3 | Baseline ML & Advanced ML**

**Goal:** Classify infant cry audio into 5 states: `hungry` · `pain` · `discomfort` · `burping` · `tired`

**What we build:**
1. Feature extraction from audio
2. Decision Tree (Baseline)
3. Random Forest + SVM (Advanced ML)
---

## Step 1: Install & Import

In [ ]:
!pip install librosa soundfile -q

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
import warnings
import os
import joblib
from pathlib import Path

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

warnings.filterwarnings('ignore')
os.makedirs('data/raw', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)

LABELS = ['hungry', 'pain', 'discomfort', 'burping', 'tired']
COLORS = ['#2196F3', '#F44336', '#FF9800', '#4CAF50', '#9C27B0']

print('Ready')

## Step 2: Create the Dataset

We generate 1,000 synthetic audio clips — 200 per class.

Each class has specific acoustic properties based on real infant cry research:
- **Pain** → highest pitch (600 Hz), very harsh
- **Hungry** → medium pitch (400 Hz), rhythmic
- **Burping** → lowest pitch (250 Hz), fast rhythm, clean

> **Real data:** Replace `data/raw/` with the [donateacry-corpus](https://github.com/gveres/donateacry-corpus) and skip this step.

In [ ]:
SAMPLE_RATE = 22050
DURATION    = 3.0

# Acoustic properties per class (from Verduzco-Flores et al. 2012)
CRY_PARAMS = {
    'hungry':     {'pitch': 400, 'pitch_spread': 60,  'rhythm': 1.2, 'noise': 0.3},
    'pain':       {'pitch': 600, 'pitch_spread': 200, 'rhythm': 0.5, 'noise': 0.8},
    'discomfort': {'pitch': 350, 'pitch_spread': 80,  'rhythm': 0.9, 'noise': 0.5},
    'burping':    {'pitch': 250, 'pitch_spread': 40,  'rhythm': 2.0, 'noise': 0.2},
    'tired':      {'pitch': 300, 'pitch_spread': 50,  'rhythm': 1.5, 'noise': 0.4},
}

def make_cry(label, seed):
    rng    = np.random.default_rng(seed)
    p      = CRY_PARAMS[label]
    t      = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION))
    pitch  = np.clip(p['pitch'] + rng.normal(0, p['pitch_spread']), 100, 1200)

    # Build the cry: harmonics + rhythmic pulsing + noise
    signal = sum((1/h) * np.sin(2*np.pi*pitch*h*t) for h in range(1, 6))
    signal = signal * (0.5 + 0.5 * np.sin(2*np.pi*p['rhythm']*t))  # rhythm
    signal = signal + rng.normal(0, p['noise'], len(t))              # harshness
    return (signal / (np.max(np.abs(signal)) + 1e-8)).astype(np.float32)

# Generate and save all clips
seed = 0
for label in LABELS:
    Path(f'data/raw/{label}').mkdir(parents=True, exist_ok=True)
    for i in range(200):
        sf.write(f'data/raw/{label}/{i:04d}.wav', make_cry(label, seed), SAMPLE_RATE)
        seed += 1

print(f'Created {200 * len(LABELS)} audio clips')

## Step 3: Feature Extraction

Raw audio = 66,000+ numbers per clip. Too many to feed directly to a model.
We extract **88 meaningful numbers** (features) from each clip instead.

**Why 8 kHz?** Infant cries have frequencies up to ~800 Hz.
Nyquist theorem says: sample rate > 2 × max frequency → 8 kHz is enough.
That's 2.75× less data to process than the standard 22 kHz.

**Features extracted:**
- MFCCs (mean + std) — shape of the vocal tract → 40 numbers
- MFCC Delta — how fast the sound is changing → 20 numbers
- MFCC Delta-Delta — acceleration of change → 20 numbers
- Spectral features (brightness, roughness, loudness) → 8 numbers

In [ ]:
SR     = 8000   # 8 kHz is enough for infant cries
N_MFCC = 20
N_FFT  = 512
HOP    = 128

def extract_features(file_path):
    # Load audio at 8 kHz, pad/trim to exactly 3 seconds
    audio, _ = librosa.load(file_path, sr=SR, duration=3.0, mono=True)
    audio     = np.pad(audio, (0, max(0, int(SR*3.0) - len(audio))))[:int(SR*3.0)]
    audio_pre = librosa.effects.preemphasis(audio)  # boost high frequencies

    # MFCCs and their temporal derivatives
    mfcc   = librosa.feature.mfcc(y=audio_pre, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP)
    delta1 = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    # Simple spectral features
    zcr      = librosa.feature.zero_crossing_rate(audio, hop_length=HOP)[0]
    rms      = librosa.feature.rms(y=audio, hop_length=HOP)[0]
    centroid = librosa.feature.spectral_centroid(y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP)[0]
    mel_db   = librosa.power_to_db(
                   librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=32,
                                                   n_fft=N_FFT, hop_length=HOP))

    # Build the 88-number feature vector
    features = []
    features.extend(np.mean(mfcc,   axis=1))   # 20
    features.extend(np.std(mfcc,    axis=1))   # 20
    features.extend(np.mean(delta1, axis=1))   # 20
    features.extend(np.mean(delta2, axis=1))   # 20
    features.extend([np.mean(zcr), np.std(zcr), np.mean(rms), np.std(rms)])         # 4
    features.extend([np.mean(mel_db), np.std(mel_db), np.mean(centroid), np.std(centroid)])  # 4
    return np.array(features, dtype=np.float32)  # 88 total


# Extract features for all 1000 clips
all_paths, all_labels = [], []
for idx, label in enumerate(LABELS):
    wavs = sorted(Path('data/raw').glob(f'{label}/*.wav'))
    all_paths  += [str(w) for w in wavs]
    all_labels += [idx] * len(wavs)

print(f'Extracting features from {len(all_paths)} files...')
X = np.array([extract_features(p) for p in all_paths], dtype=np.float32)
y = np.array(all_labels, dtype=np.int32)

print(f'Done. Feature matrix shape: {X.shape}')
print(f'Each row = 1 clip | Each column = 1 feature')

## Step 4: Exploratory Data Analysis

Before training, we look at the data to understand it.
Two things we check:
1. Are the classes balanced?
2. Do the spectrograms look different per class? (visual sanity check)

In [ ]:
# Class distribution
counts = np.bincount(y)
plt.figure(figsize=(7, 4))
plt.bar(LABELS, counts, color=COLORS, edgecolor='black')
for i, c in enumerate(counts):
    plt.text(i, c + 1, str(c), ha='center', fontweight='bold')
plt.title('Class Distribution')
plt.ylabel('Number of clips')
plt.tight_layout()
plt.savefig('reports/class_distribution.png', dpi=150)
plt.show()
print(f'Balanced: {np.all(counts == counts[0])}')

In [ ]:
# Waveform + Mel-spectrogram for each class
fig, axes = plt.subplots(5, 2, figsize=(14, 14))
fig.suptitle('Waveform & Mel-Spectrogram per Class', fontsize=13, fontweight='bold')

for i, label in enumerate(LABELS):
    audio, sr_orig = librosa.load(str(sorted(Path('data/raw').glob(f'{label}/*.wav'))[0]),
                                   sr=22050, duration=3.0)

    # Waveform
    axes[i, 0].plot(np.linspace(0, 3, len(audio)), audio, color=COLORS[i], linewidth=0.5)
    axes[i, 0].set_ylabel(label, fontweight='bold')
    if i == 0: axes[i, 0].set_title('Waveform')

    # Mel-spectrogram
    mel_db = librosa.power_to_db(librosa.feature.melspectrogram(y=audio, sr=sr_orig), ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr_orig, x_axis='time', y_axis='mel',
                                    ax=axes[i, 1], cmap='magma')
    plt.colorbar(img, ax=axes[i, 1], format='%+2.0f dB')
    if i == 0: axes[i, 1].set_title('Mel-Spectrogram')

plt.tight_layout()
plt.savefig('reports/spectrograms.png', dpi=150)
plt.show()

## Step 5: Train / Test Split

We keep 20% of data aside for final testing.
`stratify=y` ensures each class has the same proportion in both sets.
We never touch the test set during training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} clips | Test: {X_test.shape[0]} clips')
print(f'Train class counts: {np.bincount(y_train)}')
print(f'Test  class counts: {np.bincount(y_test)}')

## Step 6: Baseline — Decision Tree

A Decision Tree asks yes/no questions about features to classify.
**Example:** *'Is MFCC_1 > 5.2? → yes → pain. no → check next feature...'*

We use it as the Baseline because it is **fully interpretable** — every prediction has a human-readable rule.

**Key settings:**
- `max_depth=5` — limits how deep the tree grows (prevents memorising training data)
- `class_weight='balanced'` — treats all classes equally
- 5-fold cross-validation — trains on 5 different splits to get a reliable score

In [ ]:
# Build pipeline: scale features → train tree
dt_model = Pipeline([
    ('scaler', StandardScaler()),
    ('tree',   DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=5,
        criterion='gini',
        class_weight='balanced',
        random_state=42
    ))
])

# Cross-validation first
cv_scores = cross_val_score(dt_model, X_train, y_train,
                             cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             scoring='f1_macro')
print(f'5-Fold CV F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

# Train and evaluate on test set
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

dt_acc = accuracy_score(y_test, y_pred_dt)
dt_f1  = f1_score(y_test, y_pred_dt, average='macro')

print(f'Test Accuracy: {dt_acc:.4f}')
print(f'Test Macro F1: {dt_f1:.4f}')
print(classification_report(y_test, y_pred_dt, target_names=LABELS))

joblib.dump(dt_model, 'models/decision_tree.pkl')

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title(f'Decision Tree  —  Accuracy: {dt_acc:.1%}')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

plot_tree(dt_model['tree'], max_depth=3,
          feature_names=[f'f{i}' for i in range(X.shape[1])],
          class_names=LABELS, filled=True, rounded=True, ax=axes[1], fontsize=6)
axes[1].set_title('Decision Tree Rules (top 3 levels)')

plt.tight_layout()
plt.savefig('reports/decision_tree.png', dpi=150)
plt.show()

## Step 7: Advanced ML — Random Forest

A Random Forest trains **many decision trees** on different random subsets of data,
then takes a majority vote. This fixes the main weakness of a single tree — overfitting.

**GridSearchCV** automatically tests different settings and picks the best:
- `n_estimators` → how many trees
- `max_depth` → how deep each tree grows
- `max_features` → how many features each split considers

In [ ]:
rf_model = Pipeline([
    ('scaler', StandardScaler()),
    ('rf',     RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

param_grid = {
    'rf__n_estimators':     [100, 200],
    'rf__max_depth':        [None, 15, 25],
    'rf__min_samples_leaf': [1, 3],
    'rf__max_features':     ['sqrt', 'log2'],
}

print('Searching best settings (takes 2-3 min)...')
gs_rf = GridSearchCV(rf_model, param_grid,
                      cv=StratifiedKFold(5, shuffle=True, random_state=42),
                      scoring='f1_macro', n_jobs=-1)
gs_rf.fit(X_train, y_train)

print(f'Best settings: {gs_rf.best_params_}')
print(f'Best CV F1:    {gs_rf.best_score_:.4f}')

y_pred_rf = gs_rf.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1  = f1_score(y_test, y_pred_rf, average='macro')

print(f'Test Accuracy: {rf_acc:.4f}')
print(f'Test Macro F1: {rf_f1:.4f}')
print(classification_report(y_test, y_pred_rf, target_names=LABELS))

joblib.dump(gs_rf.best_estimator_, 'models/random_forest.pkl')

In [ ]:
# Confusion matrix + Feature importance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Greens',
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title(f'Random Forest  —  Accuracy: {rf_acc:.1%}')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

importances = gs_rf.best_estimator_['rf'].feature_importances_
top20 = np.argsort(importances)[-20:]
axes[1].barh(range(20), importances[top20], color='#2196F3')
axes[1].set_yticks(range(20))
axes[1].set_yticklabels([f'Feature {i}' for i in top20], fontsize=8)
axes[1].set_title('Top 20 Most Important Features')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('reports/random_forest.png', dpi=150)
plt.show()

## Step 8: Advanced ML — SVM

SVM finds the boundary between classes with the **maximum margin**.

The **RBF kernel** maps our 88 features into a higher-dimensional space
where the classes can be separated by a straight line.

Formula: `K(x, y) = exp(-gamma × ||x - y||²)`

**Important:** SVM needs scaled features — distances between points
must be meaningful, so all features need to be on the same scale.

In [ ]:
svm_model = Pipeline([
    ('scaler', StandardScaler()),   # required for SVM
    ('svm',    SVC(kernel='rbf', class_weight='balanced',
                   probability=True, random_state=42))
])

param_grid_svm = {
    'svm__C':     [0.1, 1, 10],
    'svm__gamma': ['scale', 'auto', 0.01],
}

print('Searching best SVM settings...')
gs_svm = GridSearchCV(svm_model, param_grid_svm,
                       cv=StratifiedKFold(5, shuffle=True, random_state=42),
                       scoring='f1_macro', n_jobs=-1)
gs_svm.fit(X_train, y_train)

print(f'Best settings: {gs_svm.best_params_}')
print(f'Best CV F1:    {gs_svm.best_score_:.4f}')

y_pred_svm = gs_svm.predict(X_test)
svm_acc = accuracy_score(y_test, y_pred_svm)
svm_f1  = f1_score(y_test, y_pred_svm, average='macro')

print(f'Test Accuracy: {svm_acc:.4f}')
print(f'Test Macro F1: {svm_f1:.4f}')
print(classification_report(y_test, y_pred_svm, target_names=LABELS))

joblib.dump(gs_svm.best_estimator_, 'models/svm.pkl')

In [ ]:
# SVM Confusion matrix
plt.figure(figsize=(7, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt='d', cmap='Purples',
            xticklabels=LABELS, yticklabels=LABELS)
plt.title(f'SVM (RBF)  —  Accuracy: {svm_acc:.1%}')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig('reports/svm.png', dpi=150)
plt.show()

## Step 9: Final Results

Comparing all three models side by side.

In [ ]:
results = {
    'Decision Tree': {'acc': dt_acc,  'f1': dt_f1},
    'Random Forest': {'acc': rf_acc,  'f1': rf_f1},
    'SVM (RBF)':     {'acc': svm_acc, 'f1': svm_f1},
}

# Print table
print(f"{'Model':<20} {'Accuracy':>10} {'Macro F1':>10}")
print('-' * 42)
for name, r in results.items():
    best = ' <- best' if r['f1'] == max(v['f1'] for v in results.values()) else ''
    print(f"{name:<20} {r['acc']:>10.1%} {r['f1']:>10.4f}{best}")

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
names = list(results.keys())
accs  = [results[m]['acc'] for m in names]
f1s   = [results[m]['f1']  for m in names]
x, w  = np.arange(len(names)), 0.35

bars_a = ax.bar(x - w/2, accs, w, label='Accuracy', color='#2196F3', alpha=0.85)
bars_f = ax.bar(x + w/2, f1s,  w, label='Macro F1',  color='#4CAF50', alpha=0.85)

for bar, v in list(zip(bars_a, accs)) + list(zip(bars_f, f1s)):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
            f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

ax.axhline(0.2, color='red', linestyle='--', alpha=0.4, label='Random chance (5 classes)')
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Phase 1 — Model Comparison', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('reports/comparison.png', dpi=150)
plt.show()

## Step 10: Download Results

In [ ]:
import shutil
shutil.make_archive('reports', 'zip', 'reports')
shutil.make_archive('models',  'zip', 'models')

from google.colab import files
files.download('reports.zip')
files.download('models.zip')
print('Done!')